In [2]:
!pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 4.9 MB/s eta 0:00:00


In [26]:
"""
AI-Based Summary Evaluation System
===================================
This notebook implements an autonomous AI evaluator that grades student-written
lecture summaries by comparing them against the original lecture transcript.

IMPORTANT: Before submission, rename this file to:
    Summary_Evaluation_Rollnumber.ipynb
    (Replace 'Rollnumber' with your actual roll number, e.g., Summary_Evaluation_CS22B1024.ipynb)

Evaluation Criteria:
- Coverage: How well the summary captures key concepts from the transcript
- Length: Appropriate summary length relative to transcript
- Clarity: Language clarity and sentence structure
- Coherence: Logical flow and connection between sentences
- Completeness: Coverage of important topics

Author: [Your Name]
Roll Number: [Your Roll Number]
Date: November 2025
"""

# ============================================================================
# IMPORTS
# ============================================================================
import os
import re
import numpy as np
import pandas as pd
from pathlib import Path
from docx import Document
from collections import Counter
import zipfile
import tempfile
import shutil
import random
import json
import warnings
import time
import traceback

# Suppress warnings
warnings.filterwarnings('ignore')

# Try to import requests for API calls
try:
    import requests
    REQUESTS_AVAILABLE = True
except ImportError:
    REQUESTS_AVAILABLE = False

# ============================================================================
# ⚙️ CONFIGURATION
# ============================================================================

# --- File Paths ---
# Option 1 (RECOMMENDED): Provide a zip file containing both files
# The zip should contain:
#   - A .docx file (transcript)
#   - An .xlsx file (summaries)
ZIP_FILE_PATH = "7Aug2025 T+S(Agentic AI).zip"

# Option 2: Or provide individual file paths (if not using zip)
transcript_path = None  # Set to "lecture_transcript.docx" if not using zip
summaries_path = None   # Set to "student_summaries.xlsx" if not using zip

# Option 3: Auto-detect (if files are in current directory with standard names)
# Leave all as None - will search for "lecture_transcript.docx" and
# "student_summaries.xlsx" in current directory

# --- Output Settings ---
# Set to None to auto-detect from data, or set manually like "CS22B1024"
ROLL_NUMBER = None

# --- LLM Integration ---
USE_LLM = True  # Set to False to use rule-based explanations
LLM_PROVIDER = "groq"  # Using Groq API

# Set your Groq API key here (as requested, this is unchanged)
GROQ_API_KEY = "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"

# --- Text Processing Constants ---
STOP_WORDS = {
    'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of',
    'with', 'is', 'are', 'was', 'be', 'been', 'by', 'from', 'as', 'it', 'this',
    'that', 'these', 'those', 'i', 'you', 'he', 'she', 'we', 'they'
}

# ============================================================================
# 📁 1. FILE I/O FUNCTIONS
# ============================================================================

def extract_zip_file(zip_path):
    """
    Extract zip file and find transcript and summaries files.
    Returns tuple: (transcript_path, summaries_path, temp_dir_path)
    """
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"Zip file not found: {zip_path}")

    # Create temporary directory for extraction
    temp_dir_path = tempfile.mkdtemp(prefix="summary_eval_")
    print(f"Extracting zip file to temporary directory: {temp_dir_path}")

    # Extract all files
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(temp_dir_path)

    # Find transcript file (look for .docx files)
    found_transcript_path = None
    found_summaries_path = None

    # Search for files with keywords
    for root, dirs, files in os.walk(temp_dir_path):
        for file in files:
            file_path = os.path.join(root, file)
            file_lower = file.lower()

            # Look for transcript
            if file_lower.endswith('.docx') and ('transcript' in file_lower or 'lecture' in file_lower):
                if found_transcript_path is None:
                    found_transcript_path = file_path
                    print(f"Found transcript: {file}")

            # Look for summaries
            if file_lower.endswith('.xlsx') and ('summary' in file_lower or 'student' in file_lower or 'attendance' in file_lower):
                if found_summaries_path is None:
                    found_summaries_path = file_path
                    print(f"Found summaries: {file}")

    # If not found by name, use first .docx and .xlsx found
    if found_transcript_path is None:
        for root, dirs, files in os.walk(temp_dir_path):
            for file in files:
                if file.lower().endswith('.docx'):
                    found_transcript_path = os.path.join(root, file)
                    print(f"Using first .docx found as transcript: {file}")
                    break
            if found_transcript_path:
                break

    if found_summaries_path is None:
        for root, dirs, files in os.walk(temp_dir_path):
            for file in files:
                if file.lower().endswith('.xlsx'):
                    found_summaries_path = os.path.join(root, file)
                    print(f"Using first .xlsx found as summaries: {file}")
                    break
            if found_summaries_path:
                break

    if not found_transcript_path:
        raise FileNotFoundError("Could not find transcript file (.docx) in zip file")
    if not found_summaries_path:
        raise FileNotFoundError("Could not find summaries file (.xlsx) in zip file")

    return found_transcript_path, found_summaries_path, temp_dir_path

def load_transcript(path):
    """
    Load transcript from .docx file.
    """
    try:
        if os.path.exists(path):
            doc = Document(path)
            text = " ".join([p.text for p in doc.paragraphs if p.text.strip()])
        else:
            raise FileNotFoundError(f"Transcript file not found: {path}")
    except Exception as e:
        raise Exception(f"Error loading transcript: {e}")
    return text if text else ""

def load_summaries(path):
    """
    Load summaries from .xlsx file.
    """
    try:
        if not os.path.exists(path):
             raise FileNotFoundError(f"Summaries file not found: {path}")

        df = pd.read_excel(path)
        # Try to find the summary column by various names
        summary_col = None
        possible_names = [
            'Summary', 'summary', 'Summary Text', 'Summary Content',
            'Student Summary', 'Summary of Lecture', 'Summary of Content',
            'Text', 'Content', 'Response', 'Answer'
        ]

        # Remove duplicates
        possible_names = list(dict.fromkeys(possible_names))

        for name in possible_names:
            if name in df.columns:
                summary_col = name
                break

        # If not found, look for columns with 'summary' in the name (case insensitive)
        if summary_col is None:
            for col in df.columns:
                if 'summary' in str(col).lower():
                    summary_col = col
                    break

        # If still not found, use the last column (often where summaries are)
        if summary_col is None and len(df.columns) > 0:
            last_col = df.columns[-1]
            if df[last_col].dtype == 'object':  # Text column
                summary_col = last_col
                print(f"Warning: Using last column '{summary_col}' as summary column")

        # Rename the found column to 'Summary' for consistency
        if summary_col and summary_col != 'Summary':
            df = df.rename(columns={summary_col: 'Summary'})
            print(f"Found summary column: '{summary_col}' (renamed to 'Summary')")
        elif summary_col == 'Summary':
             print(f"Found summary column: 'Summary'")

        # Ensure required columns exist
        required_cols = ['Email Address', 'Name of the Student', 'Roll Number', 'Institute', 'Summary']
        for col in required_cols:
            if col not in df.columns:
                df[col] = ""
        return df

    except Exception as e:
        print(f"Error loading summaries: {e}")
        return pd.DataFrame(columns=['Email Address', 'Name of the Student', 'Roll Number', 'Institute', 'Summary'])

# ============================================================================
# 🧹 2. TEXT PREPROCESSING FUNCTIONS
# ============================================================================

def preprocess_text(text):
    """
    Preprocess text for analysis: normalize whitespace, remove special characters,
    and convert to lowercase for consistent comparison.
    """
    try:
        # Handle NaN, None, and empty values
        if pd.isna(text) or text is None:
            return ""
        text = str(text).strip()
        if not text or text.lower() == 'nan':
            return ""
        text = re.sub(r'\s+', ' ', text)  # Normalize whitespace
        text = re.sub(r'[^\w\s\.!?\-]', '', text)  # Remove special chars
        return text.lower()
    except Exception:
        return ""

def split_into_sentences(text):
    """
    Split text into sentences for coherence analysis.
    """
    try:
        # Handle NaN, None, and empty values
        if pd.isna(text) or text is None:
            return []
        text = str(text).strip()
        if not text or text.lower() == 'nan':
            return []
        sentences = re.split(r'[.!?]+', text)
        return [s.strip() for s in sentences if s.strip()]
    except Exception:
        return []

def tokenize(text):
    """
    Tokenize text into words, removing stop words and short words.
    Uses the global STOP_WORDS set.
    """
    try:
        # Handle empty or None values
        if not text or pd.isna(text):
            return []
        text = str(text).lower().strip()
        if not text or text == 'nan':
            return []
        words = re.findall(r'\b\w+\b', text)
        return [w for w in words if w not in STOP_WORDS and len(w) > 2]
    except Exception:
        return []

# ============================================================================
# 📊 3. SIMILARITY & CONTENT ANALYSIS FUNCTIONS
# ============================================================================

def get_word_freq(text):
    """Returns a Counter object of word frequencies."""
    tokens = tokenize(text)
    return Counter(tokens)

def jaccard_similarity(text1, text2):
    """
    Calculate a combined similarity: 70% Jaccard + 30% Overlap Ratio.
    Jaccard = |intersection| / |union|
    Overlap = |intersection| / |set1|
    Returns a score between 0 and 1.
    """
    set1 = set(tokenize(text1))
    set2 = set(tokenize(text2))

    if not set1 or not set2:
        return 0.0

    intersection = len(set1 & set2)
    union = len(set1 | set2)
    jaccard = intersection / union if union > 0 else 0.0

    # Calculate overlap ratio (more lenient)
    overlap_ratio = intersection / len(set1) if set1 else 0.0

    # Use a combination - gives partial credit even with low Jaccard
    combined = (jaccard * 0.7) + (overlap_ratio * 0.3)
    return combined

def get_similarity(text1, text2):
    """
    Enhanced similarity function combining Jaccard and overlap metrics.
    Provides more lenient scoring for partial matches.
    """
    similarity = jaccard_similarity(text1, text2)

    # Ensure minimum similarity for any overlap - more generous
    tokens1 = set(tokenize(text1))
    tokens2 = set(tokenize(text2))

    if tokens1 and tokens2:
        overlap = len(tokens1 & tokens2)
        if overlap > 0:
            # More generous minimum similarity
            min_sim = min(0.25, overlap / max(len(tokens1), 10))  # At least 0.25 or proportional
            similarity = max(similarity, min_sim)
        # Boost similarity if there's substantial overlap
        if overlap > 5:
            overlap_ratio = overlap / len(tokens1)
            similarity = max(similarity, overlap_ratio * 0.8)  # Boost based on overlap

    return max(0.0, min(1.0, similarity))

def extract_key_concepts(text, top_n=15):
    """
    Extract key concepts/phrases from text for detailed analysis.
    """
    tokens = tokenize(text)
    if not tokens:
        return []

    # Get most frequent meaningful words (excluding very common ones)
    word_freq = Counter(tokens)
    max_freq = max(word_freq.values()) if word_freq else 1
    threshold = max(2, max_freq // 10)

    key_words = [word for word, freq in word_freq.most_common(top_n * 2)
                 if freq >= threshold and len(word) > 3]

    # Also extract noun phrases (2-3 word combinations)
    words_list = text.lower().split()
    phrases = []
    for i in range(len(words_list) - 1):
        if len(words_list[i]) > 3 and len(words_list[i+1]) > 3:
            phrases.append(f"{words_list[i]} {words_list[i+1]}")

    phrase_freq = Counter(phrases)
    key_phrases = [p for p, f in phrase_freq.most_common(top_n) if f >= 2]

    return list(set(key_words[:top_n] + key_phrases[:top_n]))

def analyze_summary_content(summary_text, summary_clean, transcript_clean, scores):
    """
    Deep content analysis to generate specific, personalized feedback.
    """
    # Extract key concepts from both
    summary_concepts = extract_key_concepts(summary_clean, top_n=20)
    transcript_concepts = extract_key_concepts(transcript_clean, top_n=30)

    # Find what's covered and what's missing
    covered_concepts = [c for c in summary_concepts if c in transcript_concepts]
    missing_concepts = [c for c in transcript_concepts[:15] if c not in summary_concepts]

    # Analyze sentence structure
    sentences = split_into_sentences(summary_clean)
    avg_sentence_length = np.mean([len(s.split()) for s in sentences]) if sentences else 0

    # Count specific details (numbers, examples, etc.)
    has_numbers = bool(re.search(r'\d+', summary_text))
    has_examples = any(word in summary_clean for word in ['example', 'instance', 'such as', 'like', 'including'])

    return {
        'covered_concepts': covered_concepts[:5],
        'missing_concepts': missing_concepts[:5],
        'sentence_count': len(sentences),
        'avg_sentence_length': avg_sentence_length,
        'has_numbers': has_numbers,
        'has_examples': has_examples,
        'word_count': len(summary_text.split())
    }

# ============================================================================
# 💯 4. EVALUATION & SCORING FUNCTIONS
# ============================================================================

def calculate_coverage(summary_clean, transcript_clean):
    """
    Calculate how well the summary covers the transcript content.
    Uses Jaccard similarity and word overlap to measure content alignment.
    Returns a score between 0 and 1.
    """
    # Calculate overall similarity
    overall_sim = get_similarity(summary_clean, transcript_clean)

    # Also check for key phrase matches (more lenient)
    summary_tokens = set(tokenize(summary_clean))
    transcript_tokens = set(tokenize(transcript_clean))

    if summary_tokens and transcript_tokens:
        # Calculate what percentage of summary words appear in transcript
        matching_words = len(summary_tokens & transcript_tokens)
        coverage_ratio = matching_words / len(summary_tokens) if summary_tokens else 0.0

        # More generous: weight coverage ratio more heavily
        combined_coverage = (overall_sim * 0.4) + (coverage_ratio * 0.6)

        # Boost coverage if there's substantial word overlap
        if matching_words > 10:
            boost = min(0.2, matching_words / 100)  # Up to 0.2 boost
            combined_coverage = min(1.0, combined_coverage + boost)

        return float(combined_coverage)

    # Even if no tokens match, give base score for non-empty summary
    return float(max(0.2, overall_sim))

def calculate_length_ratio(summary_text, transcript_text):
    """
    Evaluate if the summary length is appropriate relative to the transcript.
    Good summaries are typically 3-5% of the original length.
    Returns a ratio score and status string.

    Note: Scoring is generous to reward effort.
    """
    try:
        # Handle empty or None values
        if not summary_text or pd.isna(summary_text):
            return 0.4, "too_short"
        summary_text = str(summary_text).strip()
        if not summary_text:
            return 0.4, "too_short"

        summary_words = len(summary_text.split())
        transcript_words = len(str(transcript_text).split())
        ratio = summary_words / max(1, transcript_words)

        # More generous scoring - give higher base scores
        if ratio < 0.01:
            return 0.4, "too_short"  # Very short but still give decent points
        elif ratio < 0.03:
            return 0.7, "short"  # Short but acceptable - give good score
        elif ratio < 0.05:
            return 0.85, "adequate"  # Adequate length
        elif ratio > 0.3:
            return 0.9, "too_long"  # Very long but still good
        else:
            return 1.0, "appropriate"
    except Exception:
        return 0.5, "too_short"

def calculate_clarity(text):
    """
    Assess language clarity based on average words per sentence.
    Optimal range is typically 15-20 words per sentence.
    Returns a score between 0 and 1.

    Note: Scoring is generous.
    """
    try:
        # Handle empty or None values
        if not text or pd.isna(text):
            return 0.7
        text = str(text).strip()
        if not text:
            return 0.7

        words = len(text.split())
        sentences = len(re.split(r'[.!?]+', text))
        avg_words_per_sentence = words / max(1, sentences)

        # More generous - give higher base scores
        if avg_words_per_sentence > 30:
            return 0.75  # Very long sentences but still decent
        elif avg_words_per_sentence > 20:
            return 0.85  # Long sentences
        elif avg_words_per_sentence < 4:
            return 0.8  # Short sentences but clear
        else:
            return 1.0  # Good sentence length
    except Exception:
        return 0.75

def calculate_coherence(sentences):
    """
    Measure how well sentences flow together logically.
    Calculates similarity between consecutive sentences.
    Returns a score between 0 and 1.

    Note: Scoring is generous.
    """
    if len(sentences) < 2:
        return 0.7  # Single sentence - give decent score

    coherence_scores = []
    for i in range(len(sentences) - 1):
        if sentences[i].strip() and sentences[i+1].strip():
            sim = get_similarity(sentences[i], sentences[i+1])
            coherence_scores.append(sim)

    base_coherence = float(np.mean(coherence_scores)) if coherence_scores else 0.7
    # Boost coherence score - even low coherence gets decent score
    return max(0.6, base_coherence * 1.2) if base_coherence > 0 else 0.7

def evaluate_summary(summary_text, summary_clean, summary_sentences, transcript_clean, transcript):
    """
    Main evaluation function that combines all metrics.
    Returns a dictionary with scores for each criterion.
    Total possible score: 10.0
    (coverage: 3.0, length: 2.5, clarity: 2.0, coherence: 1.5, completeness: 1.0)

    Note: This function contains several "generous" boosts to reward student effort.
    """
    # Handle empty summaries
    if not summary_clean or len(summary_clean.strip()) == 0:
        return {
            'coverage': 0.0, 'length': 0.0, 'clarity': 0.0,
            'coherence': 0.0, 'completeness': 0.0
        }

    coverage = calculate_coverage(summary_clean, transcript_clean)
    length_ratio, length_status = calculate_length_ratio(summary_text, transcript)
    clarity = calculate_clarity(summary_clean)
    coherence = calculate_coherence(summary_sentences) if len(summary_sentences) > 1 else 0.7

    # Boost coverage score - ensure minimum coverage for any non-empty summary
    coverage_boosted = max(0.3, coverage)  # Minimum 0.3 coverage for any summary
    if coverage > 0.1:
        coverage_boosted = min(1.0, coverage * 1.3)  # Boost existing coverage

    # Calculate completeness more generously
    transcript_sentence_count = len(split_into_sentences(transcript_clean))
    completeness_ratio = len(summary_sentences) / max(1, transcript_sentence_count * 0.12)
    completeness_score = min(1.0, completeness_ratio * 1.2)  # Boost completeness

    # Increased weights and more generous scoring
    scores = {
        'coverage': max(0, min(1, coverage_boosted)) * 3.0,  # Weight: 3.0
        'length': length_ratio * 2.5,                        # Weight: 2.0
        'clarity': clarity * 2.0,                            # Weight: 2.0
        'coherence': coherence * 1.5,                        # Weight: 1.5
        'completeness': completeness_score * 1.0             # Weight: 1.0
    }

    # Ensure minimum total score of 3.0 for any non-empty summary
    total_before_min = sum(scores.values())
    if total_before_min > 0:
        if total_before_min < 3.0:
            # Scale up to ensure at least 3.0
            scale_factor = 3.0 / total_before_min
            for key in scores:
                scores[key] = scores[key] * scale_factor

        # Add bonus for longer summaries (encourage effort)
        summary_words = len(summary_text.split())
        if summary_words > 100:
            bonus = min(0.5, (summary_words - 100) / 500)  # Up to 0.5 bonus
            scores['coverage'] += bonus

    return scores

def compute_final_score(scores):
    """
    Compute final score out of 10.0 by summing all metric scores.
    Ensures minimum score of 4.0 for any non-empty summary to reward effort.
    """
    total = sum(scores.values())

    # Ensure minimum score of 4.0 for any non-empty summary
    if total > 0:
        total = max(4.0, total)

    final_score = min(10.0, max(0.0, total))
    return round(final_score, 1)

# ============================================================================
# ✍️ 5. FEEDBACK GENERATION FUNCTIONS (LLM & RULE-BASED)
# ============================================================================

def generate_llm_explanation(summary_text, transcript_text, scores):
    """
    Generate explanation using Groq API for natural, intelligent feedback.
    Groq provides fast LLM inference.
    """
    global USE_LLM, REQUESTS_AVAILABLE, GROQ_API_KEY

    if not USE_LLM or not REQUESTS_AVAILABLE or not GROQ_API_KEY:
        return None

    try:
        # Prepare the prompt
        total_score = sum(scores.values())
        coverage = scores.get('coverage', 0) / 3.0 * 100  # Convert to percentage
        length = scores.get('length', 0) / 2.5 * 100
        clarity = scores.get('clarity', 0) / 2.0 * 100
        coherence = scores.get('coherence', 0) / 1.5 * 100

        # Truncate for API limits (keep it reasonable)
        summary_preview = summary_text[:1000] if len(summary_text) > 1000 else summary_text
        transcript_preview = transcript_text[:2000] if len(transcript_text) > 2000 else transcript_text

        prompt = f"""You are an expert evaluator reviewing a student's summary of a lecture transcript.

LECTURE TRANSCRIPT (excerpt):
{transcript_preview}

STUDENT SUMMARY:
{summary_preview}

EVALUATION SCORES (out of 10.0):
- Overall Score: {total_score:.1f}/10.0
- Coverage: {coverage:.0f}% (how well it captures lecture content)
- Length: {length:.0f}% (appropriate length)
- Clarity: {clarity:.0f}% (language clarity)
- Coherence: {coherence:.0f}% (logical flow)

Write a 2-3 sentence review that:
1. Specifically mentions what concepts/topics the summary covers well or misses
2. Comments on the length and structure
3. Provides constructive feedback on clarity and coherence
4. Be specific, natural, and helpful - like a real teacher's feedback

Be concise, specific, and avoid generic phrases. Mention actual content when possible.

Review:"""

        # Call Groq API
        url = "https://api.groq.com/openai/v1/chat/completions"
        headers = {
            "Authorization": f"Bearer {GROQ_API_KEY}",
            "Content-Type": "application/json"
        }

        payload = {
            # *** UPDATED MODEL as requested ***
            "model": "llama-3.1-70b-versatile",
            "messages": [
                {"role": "system", "content": "You are an expert educator providing constructive feedback on student summaries. Be specific, helpful, and natural. Write like a real teacher reviewing student work."},
                {"role": "user", "content": prompt}
            ],
            "max_tokens": 250,
            "temperature": 0.8
        }

        response = requests.post(url, headers=headers, json=payload, timeout=30)
        response.raise_for_status()

        result = response.json()
        explanation = result['choices'][0]['message']['content'].strip()

        # Small delay to avoid rate limits
        time.sleep(0.05)  # Groq is fast, minimal delay needed

        return explanation

    except Exception as e:
        return None

def generate_explanation(scores, summary_text="", summary_clean="", transcript_clean="", transcript=""):
    """
    Generate detailed, personalized review based on actual content analysis or LLM.
    Provides specific feedback about what's covered, what's missing, and how to improve.
    """
    # Try LLM first if enabled
    if USE_LLM and summary_text and transcript:
        llm_explanation = generate_llm_explanation(summary_text, transcript, scores)
        if llm_explanation:
            return llm_explanation
        # Fall through to rule-based if LLM fails

    # --- Rule-Based Fallback ---
    coverage_score = scores['coverage']
    length_score = scores['length']
    clarity_score = scores['clarity']
    coherence_score = scores['coherence']
    completeness_score = scores.get('completeness', 0)
    total_score = sum(scores.values())

    # Perform deep content analysis if summary text is provided
    content_analysis = None
    if summary_text and summary_clean and transcript_clean:
        try:
            content_analysis = analyze_summary_content(summary_text, summary_clean, transcript_clean, scores)
        except Exception:
            content_analysis = None

    sentences = []

    # Build detailed explanation based on content analysis
    if content_analysis:
        # First sentence: Specific coverage assessment
        covered = content_analysis['covered_concepts']
        missing = content_analysis['missing_concepts']

        if coverage_score > 2.5:
            if covered:
                concepts_str = ", ".join(covered[:3])
                sentences.append(f"The summary effectively covers key concepts such as {concepts_str}, demonstrating good understanding of the lecture's main themes.")
            else:
                sentences.append("The summary demonstrates strong coverage of the lecture content, capturing essential arguments and key points effectively.")
        elif coverage_score > 2.0:
            if covered:
                concepts_str = ", ".join(covered[:2])
                if missing:
                    missing_str = ", ".join(missing[:2])
                    sentences.append(f"While the summary addresses important topics like {concepts_str}, it misses some key concepts such as {missing_str} that would strengthen the analysis.")
                else:
                    sentences.append(f"The summary covers main topics including {concepts_str}, though some nuanced details and supporting examples could enhance the depth.")
            else:
                sentences.append("The summary captures most key concepts from the lecture, though some important supporting details and examples are missing.")
        elif coverage_score > 1.3:
            if missing:
                missing_str = ", ".join(missing[:2])
                sentences.append(f"The summary touches on several topics but overlooks important concepts like {missing_str}, limiting its comprehensiveness.")
            else:
                sentences.append("While the summary addresses some main topics, it lacks sufficient depth and omits crucial information from the lecture.")
        else:
            if missing:
                missing_str = ", ".join(missing[:2])
                sentences.append(f"The summary shows limited engagement with the lecture material, missing critical concepts such as {missing_str} and other fundamental points.")
            else:
                sentences.append("The summary demonstrates minimal alignment with the lecture content, missing most key concepts and arguments.")

        # Second sentence: Length and structure with specific details
        word_count = content_analysis['word_count']
        sentence_count = content_analysis['sentence_count']

        if length_score >= 2.2:
            sentences.append(f"At {word_count} words across {sentence_count} sentences, the length is well-balanced, allowing for thorough coverage without unnecessary verbosity.")
        elif length_score >= 1.8:
            if word_count < 300:
                sentences.append(f"With {word_count} words, the summary is concise but could benefit from expanding on key points to provide more comprehensive coverage.")
            else:
                sentences.append(f"The {word_count}-word summary is adequate in length, though some sections could be tightened while others expanded for better balance.")
        elif length_score >= 1.2:
            if word_count < 200:
                sentences.append(f"The summary is too brief at {word_count} words; significant expansion is needed to adequately cover the lecture's key concepts and examples.")
            else:
                sentences.append(f"At {word_count} words, the summary needs improvement—consider either expanding on important concepts or condensing less critical information.")
        else:
            sentences.append(f"The summary length ({word_count} words) is insufficient for proper coverage; substantial expansion is required to represent the lecture adequately.")

        # Third sentence: Writing quality with specific observations
        avg_len = content_analysis['avg_sentence_length']
        has_examples = content_analysis['has_examples']
        has_numbers = content_analysis['has_numbers']
        quality_notes = []

        if clarity_score > 1.8:
            quality_notes.append("clear and well-structured sentences" if 12 <= avg_len <= 22 else "clear language")
        elif clarity_score > 1.4:
            quality_notes.append("generally clear language")
        else:
            quality_notes.append("language that needs clarity improvement")

        if coherence_score > 1.3:
            quality_notes.append("strong logical flow")
        elif coherence_score > 1.0:
            quality_notes.append("adequate coherence")
        else:
            quality_notes.append("transitions that need strengthening")

        if has_examples: quality_notes.append("includes relevant examples")
        if has_numbers: quality_notes.append("incorporates specific details")

        if quality_notes:
            notes_str = ", ".join(quality_notes[:2])
            sentences.append(f"The writing demonstrates {notes_str}, making the summary {('easy' if clarity_score > 1.6 else 'reasonably easy')} to follow.")
        else:
            sentences.append("Both clarity and coherence need improvement; consider revising sentence structure and strengthening connections between ideas.")

    else:
        # Fallback to generic score-based explanation
        score_hash = int(coverage_score * 100) % 4
        if coverage_score > 2.7:
            sentences.append([
                "The summary demonstrates strong understanding of the lecture's core concepts.",
                "The summary successfully captures the essential arguments and key points.",
                "The summary shows comprehensive grasp of the main topics discussed.",
                "The summary effectively synthesizes the lecture's primary themes."
            ][score_hash])
        elif coverage_score > 2.0:
            sentences.append("The summary captures most key concepts, though some nuanced details and supporting examples are missing.")
        elif coverage_score > 1.3:
            sentences.append("While the summary addresses several main topics, it overlooks important concepts and lacks depth in critical areas.")
        else:
            sentences.append("The summary shows minimal alignment with the lecture content, missing most key concepts and arguments.")

        # Length feedback
        if length_score >= 2.3:
            sentences.append("The length is optimal, balancing thoroughness with conciseness.")
        elif length_score >= 1.8:
            sentences.append("The length is generally appropriate, though slightly more detail in key areas would strengthen the summary.")
        else:
            sentences.append("The length is insufficient for proper coverage; the summary requires significant expansion.")

        # Quality feedback
        if clarity_score > 1.8 and coherence_score > 1.3:
            sentences.append("The writing is clear and well-organized, with smooth transitions that enhance readability.")
        elif clarity_score > 1.6:
            sentences.append("The language is clear, though some transitions between ideas could be smoother.")
        else:
            sentences.append("Both clarity and coherence need improvement; consider revising sentence structure.")

    # Add a concluding remark
    if len(sentences) == 2:
        if total_score >= 7.5:
            sentences.append("Overall, this is a strong summary that effectively communicates the lecture's main points.")
        elif total_score >= 6.0:
            sentences.append("With some refinement, this summary could more effectively capture the lecture's key insights.")
        else:
            sentences.append("Additional effort in coverage and clarity would significantly improve this summary.")

    # Combine sentences
    explanation = ". ".join(sentences[:3]) # Max 3 sentences
    if not explanation.endswith('.'):
        explanation += "."

    return explanation

# ============================================================================
# 🚀 6. MAIN EXECUTION PIPELINE
# ============================================================================

def run_evaluation_pipeline():
    """
    Main function to run the entire evaluation process.
    """
    global transcript_path, summaries_path, ROLL_NUMBER, USE_LLM, REQUESTS_AVAILABLE, GROQ_API_KEY

    print("=" * 60)
    print("AI-Based Summary Evaluation System")
    print("=" * 60)

    # --- LLM Setup Check ---
    if USE_LLM:
        print("\n📝 LLM Mode: ENABLED")
        if not REQUESTS_AVAILABLE:
            print("⚠ 'requests' library not installed. Install with: pip install requests")
            print("   Falling back to rule-based explanations.")
            USE_LLM = False
        elif not GROQ_API_KEY:
            print("⚠ Groq API key not set. Set GROQ_API_KEY in the configuration.")
            print("   Falling back to rule-based explanations.")
            USE_LLM = False
        else:
            print(f"   Using Groq with model: llama-3.1-405b-reasoning")
            print("   Get key from: https://console.groq.com/")
    else:
        print("\n📝 LLM Mode: DISABLED")
        print("   Using rule-based explanations")


    temp_dir_path = None
    output_path = None

    try:
        # ==================================
        # STEP 1: Load Input Files
        # ==================================
        print("\n[STEP 1] Loading input files...")

        # Handle zip file or individual files
        if ZIP_FILE_PATH:
            print(f"Processing zip file: {ZIP_FILE_PATH}")
            transcript_path, summaries_path, temp_dir_path = extract_zip_file(ZIP_FILE_PATH)
            print("✓ Files extracted successfully")
        elif transcript_path and summaries_path:
            print("Using individual file paths")
        else:
            # Try to find files automatically
            print("Searching for input files in current directory...")
            if os.path.exists("lecture_transcript.docx"):
                transcript_path = "lecture_transcript.docx"
            if os.path.exists("student_summaries.xlsx"):
                summaries_path = "student_summaries.xlsx"

            if not transcript_path or not summaries_path:
                raise FileNotFoundError(
                    "Please provide either:\n"
                    "  1. ZIP_FILE_PATH = 'your_file.zip' (containing transcript and summaries), OR\n"
                    "  2. transcript_path = 'lecture_transcript.docx' and summaries_path = 'student_summaries.xlsx'"
                )

        # Load the files
        transcript = load_transcript(transcript_path)
        summaries_df = load_summaries(summaries_path)

        if len(summaries_df) == 0:
            print("ERROR: No summaries file found or file is empty.")
            raise FileNotFoundError(f"Could not load summaries from {summaries_path}")

        print(f"✓ Transcript loaded: {len(transcript)} characters")
        print(f"✓ Summaries loaded: {len(summaries_df)} students")

        # Auto-detect roll number from first student if not set
        if ROLL_NUMBER is None and len(summaries_df) > 0:
            first_roll = str(summaries_df.iloc[0].get('Roll Number', '')).strip()
            ROLL_NUMBER = first_roll.upper() if first_roll else "UNKNOWN"
            print(f"✓ Auto-detected Roll Number: {ROLL_NUMBER}")

        # Set output path
        output_path = f"Graded_results_{ROLL_NUMBER}.xlsx"
        print(f"✓ Output file will be: {output_path}")

        # ==================================
        # STEP 2: Text Preprocessing
        # ==================================
        print("\n[STEP 2] Preprocessing text...")

        transcript_clean = preprocess_text(transcript)
        print(f"✓ Transcript processed.")

        # Handle NaN values properly
        summaries_df['Summary'] = summaries_df['Summary'].fillna('')

        # Check if Summary column is actually empty - maybe it's in a different column
        summary_lengths = summaries_df['Summary'].apply(lambda x: len(str(x).strip()) if pd.notna(x) else 0)
        if summary_lengths.sum() == 0:
            print("WARNING: 'Summary' column appears empty! Checking other columns...")
            for col in summaries_df.columns:
                if col not in ['Summary', 'Email Address', 'Name of the Student', 'Roll Number', 'Institute']:
                    col_lengths = summaries_df[col].apply(lambda x: len(str(x).strip()) if pd.notna(x) else 0)
                    non_empty = (col_lengths > 10).sum()  # More than 10 chars
                    if non_empty > len(summaries_df) * 0.5 and col_lengths.mean() > 50:
                        print(f"   -> Using column '{col}' as Summary column!")
                        summaries_df['Summary'] = summaries_df[col]
                        break

        summaries_df['Summary_clean'] = summaries_df['Summary'].apply(preprocess_text)
        summaries_df['Summary_sentences'] = summaries_df['Summary'].apply(split_into_sentences)
        print("✓ Student summaries processed")

        # Debug: Check summary lengths
        summary_lengths = summaries_df['Summary'].apply(lambda x: len(str(x).strip()) if pd.notna(x) else 0)
        print(f"   Summary length stats: Min={summary_lengths.min()}, Max={summary_lengths.max()}, Mean={summary_lengths.mean():.1f}")
        print(f"   Non-empty summaries: {(summary_lengths > 0).sum()} out of {len(summaries_df)}")

        # ==================================
        # STEP 3: Evaluate Each Summary
        # ==================================
        print("\n[STEP 3] Evaluating summaries...")
        if USE_LLM:
            print("   (Using Groq LLM for explanations - this will be fast!)")
        else:
            print("   (Using rule-based explanations...)")

        results = []
        for idx, row in summaries_df.iterrows():
            try:
                summary_text = str(row.get('Summary', '') or '').strip()
                if pd.isna(row.get('Summary')) or summary_text.lower() == 'nan':
                    summary_text = ''

                summary_clean = row.get('Summary_clean', '') or ''
                summary_sentences = row.get('Summary_sentences', []) or []

                if not summary_text:
                    # Only give 0 if completely empty
                    score = 0.0
                    explanation = 'Summary is empty and cannot be evaluated.'
                else:
                    # Evaluate all non-empty summaries
                    scores = evaluate_summary(summary_text, summary_clean, summary_sentences, transcript_clean, transcript)
                    score = compute_final_score(scores)
                    explanation = generate_explanation(scores, summary_text, summary_clean, transcript_clean, transcript)

                if idx > 0 and idx % 10 == 0:  # Show progress every 10 summaries
                    print(f"   Progress: {idx}/{len(summaries_df)} summaries reviewed...")

                # Prepare output row
                results.append({
                    'Email Address': str(row.get('Email Address', '') or ''),
                    'Name': str(row.get('Name of the Student', '') or ''),
                    'Roll Number': str(row.get('Roll Number', '') or ''),
                    'Institute': str(row.get('Institute', '') or ''),
                    'Summary': summary_text[:500] if summary_text else '',  # Truncate for display
                    'Score': score,
                    'Explanation': explanation
                })
            except Exception as e:
                print(f"Error processing row {idx} ({row.get('Name of the Student')}): {str(e)}")
                traceback.print_exc()

        # Create results dataframe
        results_df = pd.DataFrame(results)
        print(f"\n✓ Evaluation complete. Total summaries: {len(results_df)}")

        # ==================================
        # STEP 4: Post-processing & Save
        # ==================================
        print("\n[STEP 4] Post-processing scores and saving results...")

        if len(results_df) > 0:
            current_mean = results_df['Score'].mean()
            print(f"   Initial score distribution: Min={results_df['Score'].min():.2f}, Max={results_df['Score'].max():.2f}, Mean={current_mean:.2f}")

            # If mean is below 5, scale up all scores proportionally (as per original logic)
            if current_mean < 5.0 and current_mean > 0:
                scale_factor = 5.0 / current_mean
                scale_factor = min(scale_factor, 1.5)  # Cap the scale factor to avoid inflating too much
                print(f"   Mean is below 5.0. Scaling scores by {scale_factor:.2f}...")
                results_df['Score'] = results_df['Score'] * scale_factor
                results_df['Score'] = results_df['Score'].clip(upper=10.0)  # Cap at 10
                results_df['Score'] = results_df['Score'].round(1)
                print(f"   Adjusted scores. New Mean={results_df['Score'].mean():.2f}")
            else:
                print(f"   Mean is {current_mean:.2f} (>= 5.0), no adjustment needed.")

        # Save to file
        try:
            results_df.to_excel(output_path, index=False, engine='openpyxl')
            print(f"\n✓ Results successfully exported to: {output_path}")
        except Exception as e:
            print(f"   Error saving to Excel: {e}. Trying CSV...")
            try:
                csv_output_path = output_path.replace('.xlsx', '.csv')
                results_df.to_csv(csv_output_path, index=False)
                print(f"\n✓ Results successfully exported to CSV: {csv_output_path}")
            except Exception as e2:
                print(f"   FATAL: Could not save results to file: {e2}")

        return results_df

    except Exception as e:
        print(f"\n--- AN ERROR OCCURRED ---")
        print(f"Error: {e}")
        traceback.print_exc()
        return None

    finally:
        # Clean up temporary extraction directory if used
        if temp_dir_path and os.path.exists(temp_dir_path):
            try:
                shutil.rmtree(temp_dir_path)
                print(f"✓ Cleaned up temporary files from {temp_dir_path}")
            except Exception as e:
                print(f"⚠ Could not clean up temporary directory: {temp_dir_path} ({e})")

# ============================================================================
# 🏁 RUN THE EVALUATOR
# ============================================================================

# Run the entire pipeline
final_results_df = run_evaluation_pipeline()

# Display final summary if successful
if final_results_df is not None and not final_results_df.empty:
    print("\n" + "=" * 60)
    print("FINAL RESULTS SUMMARY")
    print("=" * 60)

    display_cols = ['Name', 'Score', 'Explanation']
    print(final_results_df[display_cols].head(10).to_string(index=False))

    print("\nScore Statistics:")
    print(f"  Min:     {final_results_df['Score'].min():.2f}")
    print(f"  Max:     {final_results_df['Score'].max():.2f}")
    print(f"  Mean:    {final_results_df['Score'].mean():.2f}")
    print(f"  Std Dev: {final_results_df['Score'].std():.2f}")
    print(f"\n✓ Total students evaluated: {len(final_results_df)}")
else:
    print("\nEvaluation did not complete successfully. Please check errors above.")

AI-Based Summary Evaluation System

📝 LLM Mode: ENABLED
   Using Groq with model: llama-3.1-405b-reasoning
   Get key from: https://console.groq.com/

[STEP 1] Loading input files...
Processing zip file: 7Aug2025 T+S(Agentic AI).zip
Extracting zip file to temporary directory: /tmp/summary_eval_qtlxksel
Found transcript: Agentic AI Class - 2025_08_07 18_10 GMT+05_30 - Transcript.docx
Found summaries: Attendance + Summary 1.xlsx
✓ Files extracted successfully
Found summary column: 'Summary(in 100words)' (renamed to 'Summary')
✓ Transcript loaded: 61243 characters
✓ Summaries loaded: 82 students
✓ Auto-detected Roll Number: UNKNOWN
✓ Output file will be: Graded_results_UNKNOWN.xlsx

[STEP 2] Preprocessing text...
✓ Transcript processed.
✓ Student summaries processed
   Summary length stats: Min=10, Max=956, Mean=505.4
   Non-empty summaries: 82 out of 82

[STEP 3] Evaluating summaries...
   (Using Groq LLM for explanations - this will be fast!)
   Progress: 10/82 summaries reviewed...
   